# Checkpoint 3 — Inverse Problem via Emulator

**Goal:** given surface displacement measurements, recover:
- `layer_ID` — which layer contains the pressure anomaly (discrete, 0–3)
- `pressure_value` — anomaly pressure in [−0.5, 0.5]
- `center_location_mean` — cavity center (cx, cy)

**Strategy:**
1. Enumerate the 4 possible layers
2. For each layer, run L-BFGS-B with multiple random restarts over (pressure, cx, cy)
3. Pick the layer + parameters that minimise the L2 residual against the observed displacements

## 1 — Imports

In [10]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import joblib
from scipy.optimize import minimize
from scipy.interpolate import PchipInterpolator

from create_grid import create_layered_grid_with_hole
from common_functions import solve_and_export_pressure

## 2 — Forward model setup (given code)

In [11]:
folder = Path.cwd()

mesh_size = 0.025

sd, cell_layer = create_layered_grid_with_hole(
    mesh_size=mesh_size, center=(1.5, 0.25)
)

folder_export = folder / "results"
layer_ID_true = 3
pressure_true = 0.5

u = solve_and_export_pressure(
    sd,
    cell_layer,
    layer_ID_true,
    pressure_true,
    folder_export,
    export_name="sol_layered_ID3",
    cell_data=[("layer", cell_layer.astype(float))],
)

print("u[0,:] shape:", np.shape(u[0, :]))

# Extraction of surface displacements
y_coords  = sd.nodes[1, :]
idx       = np.where(np.isclose(y_coords, 1.0))[0]
top_nodes = sd.nodes[0:2, idx]   # (2, N)  rows = [x ; y]
top_u     = u[0:2, idx]          # (2, N)  rows = [ux; uy]

print("top_nodes shape :", top_nodes.shape)
print("top_u shape     :", top_u.shape)
print("x range         :", top_nodes[0].min(), "to", top_nodes[0].max())


u[0,:] shape: (6274,)
top_nodes shape : (2, 83)
top_u shape     : (2, 83)
x range         : 0.0 to 2.0


## 3 — Diagnostics: verify GP model files exist

**Run this before anything else.** If any file is missing the inversion will fail with `None`.

In [12]:
# Point to your trained GP models
MODELS_DIR = Path("/Users/edoardoventuroli/SCT4AMM/GeoMechLab/models")

# Confirm it exists before going any further
assert MODELS_DIR.exists(), f"Models folder not found: {MODELS_DIR}"
print(f"Models directory: {MODELS_DIR}")
def _load_models(layer_ID: int) -> dict:
    if layer_ID not in _gp_cache:
        ux_path = MODELS_DIR / f"gp_ux_surf_layer_{layer_ID}.pkl"
        uy_path = MODELS_DIR / f"gp_uy_surf_layer_{layer_ID}.pkl"
        if not ux_path.exists() or not uy_path.exists():
            raise FileNotFoundError(
                f"GP model files not found for layer {layer_ID}.\n"
                f"  Expected : {ux_path}\n"
                f"  Expected : {uy_path}"
            )
        _gp_cache[layer_ID] = {
            "ux": joblib.load(ux_path),
            "uy": joblib.load(uy_path),
        }
    return _gp_cache[layer_ID]
all_ok = True
for L in range(4):
    for component in ("ux", "uy"):
        p = MODELS_DIR / f"gp_{component}_surf_layer_{L}.pkl"
        status = "OK" if p.exists() else "MISSING"
        print(f"  Layer {L} | {component} | {str(p):<60} {status}")
        if not p.exists():
            all_ok = False

print()
print("All 8 model files found." if all_ok else "WARNING: missing files above.")

Models directory: /Users/edoardoventuroli/SCT4AMM/GeoMechLab/models
  Layer 0 | ux | /Users/edoardoventuroli/SCT4AMM/GeoMechLab/models/gp_ux_surf_layer_0.pkl OK
  Layer 0 | uy | /Users/edoardoventuroli/SCT4AMM/GeoMechLab/models/gp_uy_surf_layer_0.pkl OK
  Layer 1 | ux | /Users/edoardoventuroli/SCT4AMM/GeoMechLab/models/gp_ux_surf_layer_1.pkl OK
  Layer 1 | uy | /Users/edoardoventuroli/SCT4AMM/GeoMechLab/models/gp_uy_surf_layer_1.pkl OK
  Layer 2 | ux | /Users/edoardoventuroli/SCT4AMM/GeoMechLab/models/gp_ux_surf_layer_2.pkl OK
  Layer 2 | uy | /Users/edoardoventuroli/SCT4AMM/GeoMechLab/models/gp_uy_surf_layer_2.pkl OK
  Layer 3 | ux | /Users/edoardoventuroli/SCT4AMM/GeoMechLab/models/gp_ux_surf_layer_3.pkl OK
  Layer 3 | uy | /Users/edoardoventuroli/SCT4AMM/GeoMechLab/models/gp_uy_surf_layer_3.pkl OK

All 8 model files found.


## 4 — GP emulator helpers

In [13]:
_gp_cache: dict = {}


def _load_models(layer_ID: int) -> dict:
    """Load and cache GP surface-displacement models for one layer."""
    if layer_ID not in _gp_cache:
        ux_path = MODELS_DIR / f"gp_ux_surf_layer_{layer_ID}.pkl"
        uy_path = MODELS_DIR / f"gp_uy_surf_layer_{layer_ID}.pkl"
        if not ux_path.exists() or not uy_path.exists():
            raise FileNotFoundError(
                f"GP model files not found for layer {layer_ID}.\n"
                f"  Expected : {ux_path}\n"
                f"  Expected : {uy_path}"
            )
        _gp_cache[layer_ID] = {
            "ux": joblib.load(ux_path),
            "uy": joblib.load(uy_path),
        }
    return _gp_cache[layer_ID]


def _emulator_surface(layer_ID: int, pressure: float, cx: float, cy: float):
    """Return (ux_vec, uy_vec) each shape (50,) on linspace(0,2,50)."""
    m = _load_models(layer_ID)
    X = np.array([[cx, cy, pressure]])
    return m["ux"].predict(X)[0], m["uy"].predict(X)[0]


_X_EVAL = np.linspace(0.0, 2.0, 50)


def _residual(params, layer_ID, x_obs, ux_obs, uy_obs):
    """L2 residual: emulator prediction vs observations."""
    pressure, cx, cy = params
    ux_pred, uy_pred = _emulator_surface(layer_ID, pressure, cx, cy)
    f_ux = PchipInterpolator(_X_EVAL, ux_pred)
    f_uy = PchipInterpolator(_X_EVAL, uy_pred)
    return float(
        np.sum((f_ux(x_obs) - ux_obs) ** 2) +
        np.sum((f_uy(x_obs) - uy_obs) ** 2)
    )


# Sanity check
try:
    _ux_t, _uy_t = _emulator_surface(0, 0.0, 1.0, 0.5)
    print(f"Emulator test OK — layer 0: ux shape {_ux_t.shape}, uy shape {_uy_t.shape}")
except FileNotFoundError as e:
    print(f"Emulator test FAILED (model files missing):\n{e}")
except Exception as e:
    print(f"Emulator test FAILED (unexpected error):\n{e}")

Emulator test OK — layer 0: ux shape (50,), uy shape (50,)


## 5 — `GroupSolutionCP3` (submission function)

Matches the provided template exactly:
```python
def GroupSolutionCP3(top_u, top_nodes) -> (layer_ID, pressure_value, center_location_mean)
```

In [14]:
def GroupSolutionCP3(top_u, top_nodes):
    """
    Inverse problem: recover the subsurface anomaly from surface displacement.

    Parameters
    ----------
    top_u     : np.ndarray, shape (2, N) — rows = [ux; uy]
    top_nodes : np.ndarray, shape (2, N) — rows = [x ; y]

    Returns
    -------
    layer_ID             : int
    pressure_value       : float
    center_location_mean : tuple (cx, cy)
    """
    # ── Reformat inputs ───────────────────────────────────────────
    x_obs  = np.asarray(top_nodes[0, :], dtype=float)
    ux_obs = np.asarray(top_u[0, :],     dtype=float)
    uy_obs = np.asarray(top_u[1, :],     dtype=float)

    # ── Settings ──────────────────────────────────────────────────
    bounds     = [(-0.5, 0.5), (0.1, 1.9), (0.1, 0.9)]   # pressure, cx, cy
    n_restarts = 15
    rng        = np.random.default_rng(42)

    best_overall_loss = np.inf
    best_layer        = None
    best_params       = None

    # ── Stage 1: enumerate layers ─────────────────────────────────
    for L in range(4):
        # Raise immediately if model files are missing — do NOT hide this
        _load_models(L)

        layer_best_loss   = np.inf
        layer_best_params = None

        # Random starts + one centred warm start
        p_inits  = rng.uniform(-0.5, 0.5, n_restarts)
        cx_inits = rng.uniform( 0.1, 1.9, n_restarts)
        cy_inits = rng.uniform( 0.1, 0.9, n_restarts)
        p_inits[0], cx_inits[0], cy_inits[0] = 0.0, 1.0, 0.5

        # ── Stage 2: multi-start L-BFGS-B ─────────────────────────
        for p0, cx0, cy0 in zip(p_inits, cx_inits, cy_inits):
            try:
                result = minimize(
                    _residual,
                    x0     = np.array([p0, cx0, cy0]),
                    args   = (L, x_obs, ux_obs, uy_obs),
                    method = "L-BFGS-B",
                    bounds = bounds,
                    options= {"maxiter": 200, "ftol": 1e-12, "gtol": 1e-8},
                )
                if result.fun < layer_best_loss:
                    layer_best_loss   = result.fun
                    layer_best_params = result.x.copy()
            except Exception as e:
                # Only swallow numerical errors, not missing-file errors
                print(f"  [Layer {L}] optimizer run failed: {e}")
                continue

        print(f"  Layer {L}: best loss = {layer_best_loss:.4e}  "
              f"params = {layer_best_params}")

        if layer_best_params is not None and layer_best_loss < best_overall_loss:
            best_overall_loss = layer_best_loss
            best_layer        = L
            best_params       = layer_best_params

    # ── Guard against total failure ───────────────────────────────
    if best_params is None:
        raise RuntimeError(
            "Inversion failed: no optimizer run succeeded across all 4 layers.\n"
            "Re-run cell 3 to check that all GP model files are present."
        )

    layer_ID             = best_layer
    pressure_value       = float(best_params[0])
    center_location_mean = (float(best_params[1]), float(best_params[2]))

    return layer_ID, pressure_value, center_location_mean

## 6 — Smoke-test on the forward simulation

In [15]:
est_layer, est_pressure, est_center = GroupSolutionCP3(top_u, top_nodes)

print()
print(f"Estimated layer    : {est_layer}       (true = {layer_ID_true})")
print(f"Estimated pressure : {est_pressure:+.4f}  (true = {pressure_true:+.4f})")
print(f"Estimated center   : ({est_center[0]:.4f}, {est_center[1]:.4f})  "
      f"(true = (1.5000, 0.2500))")

  Layer 0: best loss = 3.6259e-01  params = [0.29722802 1.05117656 0.9       ]
  Layer 1: best loss = 2.8543e-01  params = [0.48790962 1.11223226 0.89939272]
  Layer 2: best loss = 1.4490e-01  params = [0.5        1.37539025 0.9       ]
  Layer 3: best loss = 1.4075e-05  params = [0.5        1.3298993  0.29182854]

Estimated layer    : 3       (true = 3)
Estimated pressure : +0.5000  (true = +0.5000)
Estimated center   : (1.3299, 0.2918)  (true = (1.5000, 0.2500))


## 7 — Full validation against synthetic test cases

In [16]:
from stefpy import GroupSolutionCP2


def make_synthetic_obs(layer_ID, pressure, cx, cy, x_obs, noise_std=0.0):
    """
    Synthetic observations via the CP2 emulator.
    Returns shape (2, N) matching GroupSolutionCP3's input format.
    """
    mean_surf_func, _ = GroupSolutionCP2(
        layer_ID             = layer_ID,
        pressure_value       = pressure,
        center_location_mean = (cx, cy),
        center_location_std  = (0.01, 0.01),
    )
    disp = np.array([mean_surf_func(x) for x in x_obs]).T   # (2, N)
    if noise_std > 0:
        disp += np.random.default_rng(0).normal(0, noise_std, disp.shape)
    return disp

In [17]:
test_cases = [
    {"id": 1, "layer": 3, "press":  0.50, "cx": 1.50, "cy": 0.25},
    {"id": 2, "layer": 2, "press": -0.30, "cx": 0.80, "cy": 0.50},
    {"id": 3, "layer": 0, "press":  0.40, "cx": 1.20, "cy": 0.35},
    {"id": 4, "layer": 1, "press": -0.45, "cx": 0.60, "cy": 0.60},
    {"id": 5, "layer": 3, "press":  0.20, "cx": 1.00, "cy": 0.40},
]

x_obs_val     = np.linspace(0.05, 1.95, 20)
top_nodes_val = np.vstack([x_obs_val, np.ones_like(x_obs_val)])  # (2, 20)

print("=" * 70)
print("CP3 VALIDATION")
print("=" * 70)

for tc in test_cases:
    print(f"\n{'─'*70}")
    print(f"Test {tc['id']} | True: layer={tc['layer']}, "
          f"p={tc['press']:+.2f}, cx={tc['cx']}, cy={tc['cy']}")

    top_u_synth = make_synthetic_obs(
        tc["layer"], tc["press"], tc["cx"], tc["cy"], x_obs_val
    )

    est_layer, est_p, (est_cx, est_cy) = GroupSolutionCP3(
        top_u_synth, top_nodes_val
    )

    layer_ok = est_layer == tc["layer"]
    print(f"  Layer    : est={est_layer}  true={tc['layer']}  "
          f"{'OK' if layer_ok else 'WRONG'}")
    print(f"  Pressure : est={est_p:+.4f}  true={tc['press']:+.4f}  "
          f"|err|={abs(est_p - tc['press']):.4f}")
    print(f"  cx       : est={est_cx:.4f}   true={tc['cx']:.4f}   "
          f"|err|={abs(est_cx - tc['cx']):.4f}")
    print(f"  cy       : est={est_cy:.4f}   true={tc['cy']:.4f}   "
          f"|err|={abs(est_cy - tc['cy']):.4f}")

    # Plot
    x_plot = np.linspace(0, 2, 100)
    rec_func,  _ = GroupSolutionCP2(
        layer_ID=est_layer, pressure_value=est_p,
        center_location_mean=(est_cx, est_cy), center_location_std=(0.01, 0.01),
    )
    true_func, _ = GroupSolutionCP2(
        layer_ID=tc["layer"], pressure_value=tc["press"],
        center_location_mean=(tc["cx"], tc["cy"]), center_location_std=(0.01, 0.01),
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(
        f"Test {tc['id']} — recovered layer {est_layer} "
        f"({'correct' if layer_ok else 'WRONG, true=' + str(tc['layer'])})",
        fontsize=13,
    )
    for ax, row, label in zip(axes, [0, 1], ["Ux", "Uy"]):
        ax.plot(x_plot, [true_func(x)[row] for x in x_plot],
                "r--", lw=2, label="True")
        ax.plot(x_plot, [rec_func(x)[row] for x in x_plot],
                "b-",  lw=1.5, label="Recovered")
        ax.scatter(x_obs_val, top_u_synth[row, :],
                   s=18, c="k", zorder=5, label="Observations")
        ax.set_xlabel("x (m)"); ax.set_ylabel(label)
        ax.set_title(label); ax.legend(fontsize=8); ax.grid(True)
    plt.tight_layout()
    plt.show()

CP3 VALIDATION

──────────────────────────────────────────────────────────────────────
Test 1 | True: layer=3, p=+0.50, cx=1.5, cy=0.25


FileNotFoundError: Modelli mancanti per Layer 3. Addestramento non completato?